# Alien Dictionary

```https://www.geeksforgeeks.org/problems/alien-dictionary/1```

You are given a list of words from an unknown language. The words are already sorted according to that language's dictionary order.

Your task is to return a string containing the unique characters in one valid order for that language.

If the input does not allow any valid ordering, return an empty string `""`.

A valid result must:
- include every distinct character that appears in `words` exactly once
- respect the ordering implied by the sorted word list
- return `""` when the constraints contradict each other

Notes:
- More than one valid answer may exist for some inputs.
- A longer word cannot appear before its own prefix. For example, `["abc", "ab"]` is invalid.

Example 1:
Input: words = ["wrt", "wrf", "er", "ett", "rftt"]
Output: "wertf"

Example 2:
Input: words = ["z", "x"]
Output: "zx"

Example 3:
Input: words = ["z", "x", "z"]
Output: ""

Constraints:
- `1 <= words.length <= 100`
- `1 <= words[i].length <= 100`
- `words[i]` consists of lowercase English letters


No access to leetcode and inexhaustive test cases.


In [ ]:
class Solution:
    def alienOrder(self, words: list[str]) -> str:
        # My first thought is combinatorial explosion O(n^2 k^2) with word length k and between all the possible strings "n"
        # however this first differing could also hold a transitive invariant/ equivalence class differ at position x with letters y_1, y_2 are unique.
        # Turns out alien dicitionary is a partial ordering on words, but a minimal example I can use to get total ordering of characters is have all |alphabet| - 1 examples of length 1 arranged at a chain.

        #  so my current intuition is to create a partially ordered lattice from linear iteration, 
        #  due to transitivity + minimal constraints + sorted arrays.
        #  In the sense that the sorted array + uniqueness of words in a dictionary gurantees a single differing position, 
        #  by the degrees of freedom/ solvability statement we reasoned that only the first differing mattered,
        #  my maximal information gain can be done via checking the first differing position and resolving
        #  the difference as a dependency or ordering relation. 
        #  the sortedness gives arrow direction and uniqueness + sortedness gives maximal hence this is both
        #  necessary and sufficient way to gain the best ordering understanding of our lexicography.
        #  In addition the ordering transitivity of characters is carried over to the first
        #  differing character of the words by sortedness (proveable by induction on prefix).


        # 6. Alien Dictionary through this lens

        # The algorithm is effectively:

        # Step 1

        # Extract minimal local witnesses:
        # (first differing characters)
        if len(words) == 0:
            return ""
        elif len(words) == 1:
            return words[0][0]

        graph = {} # set of set notations.
        root = None
        for i in range(1, len(words)):
            for j in range(len(words[i])):
                # find local constraint, the differing location
                prev = words[i-1][j]
                curr = words[i][j]
                    
                if prev != curr:
                    # then words[i][j] is after words[i-1][j] in the implicit lexicographic ordering constraint
                    if i == 1:         # if i == 1 then that's the minimal element in the poset.
                        root = prev
                    if prev in graph:
                        graph[prev].add(curr)
                    else:
                        graph[prev] = {curr}
        
        
        


        # Step 2

        # Construct irreducible precedence constraints

        # Step 3

        # Infer global ordering through transitive closure/topological reasoning

        # So the adjacent comparisons behave like:

        # generators of the partial order.



In [ ]:
from collections import deque
class Solution:
    def alienOrder(self, words: list[str]) -> str:
        # Extract minimal local witnesses:
        # (first differing characters)
        if len(words) == 0:
            return ""
        elif len(words) == 1:
            return words[0][0]


        # along with storing graph we can store an ordering already.
        graph = {} # set of set notations.
        indegrees = {char: 0 for word in words for char in word}
        for i in range(1, len(words)):
            for j in range(len(words[i])):
                # find local constraint, the differing location
                prev = words[i-1][j]
                curr = words[i][j]
                    
                if prev != curr:
                    print(f"i: {i}, words[i-1]: {words[i-1]}, words[i]: {words[i]}, prev: {prev}, curr: {curr}")
               
                    # then words[i][j] is after words[i-1][j] in the implicit lexicographic ordering constraint
                    # constraint seen and broken: --> this can be handled like course scheduling topo sort 
                    # later handling full deque then if graph return ""
                    # if curr in graph and prev in graph[curr]:
                    #     # invalid ordering since ≠, ≥, ≤ all at the same time.
                    #     return ""
                    # otherwise add (which should be idempotent by set construction)
                    if prev in graph:
                        graph[prev].add(curr)
                    else: #prev was not seen before
                        graph[prev] = {curr}                    
                    # adding indegrees for curr
                    indegrees[curr] += 1
                    break

        print(f"graph: {graph}")
        print(f"indegrees: {indegrees}")
        queue = deque(i for i, deg in indegrees.items() if deg == 0)
        print(f"Initial queue: {queue}")
        tentative = ""
        while queue:
            print(f"queue: {queue}")
            #iterative relaxation of the graph
            node = queue.popleft()
            tentative += node
            if node in graph:
                targets = graph[node]
                graph.pop(node)
                for target in targets:
                    indegrees[target] -= 1
                    if indegrees[target] == 0:
                        queue.append(target)

        print(f"final tentative: {tentative}, graph: {graph}")

        if len(graph) > 0:
            return ""
        else:
            return tentative

        # Step 2

        # Construct irreducible precedence constraints

        # Step 3

        # Infer global ordering through transitive closure/topological reasoning

        # So the adjacent comparisons behave like:

        # generators of the partial order.



In [47]:
def _is_valid_order(words, order):
    chars = {ch for word in words for ch in word}

    if len(order) != len(chars):
        return False
    if set(order) != chars:
        return False
    if len(set(order)) != len(order):
        return False

    pos = {ch: i for i, ch in enumerate(order)}

    for i in range(len(words) - 1):
        a, b = words[i], words[i + 1]
        for ca, cb in zip(a, b):
            if ca != cb:
                if pos[ca] > pos[cb]:
                    return False
                break
        else:
            if len(a) > len(b):
                return False

    return True


def test(solution):
    cases = [
        ("reference example", ["wrt", "wrf", "er", "ett", "rftt"], True),
        ("two-letter ordering", ["z", "x"], True),
        ("simple cycle", ["z", "x", "z"], False),
        ("invalid prefix", ["abc", "ab"], False),
        ("single word still includes every character", ["abc"], True),
        ("duplicate identical words", ["abc", "abc"], True),
        ("characters with no direct relation still appear", ["ab", "ac"], True),
        ("multiple valid answers", ["za", "zb", "ca", "cb"], True),
        ("repeated constraints remain valid", ["abc", "abx", "abx", "abz"], True),
        ("deeper chain with shared prefixes", ["baa", "abcd", "abca", "cab", "cad"], True),
        ("another prefix failure", ["xza", "xz"], False),
    ]

    for i, (name, words, should_exist) in enumerate(cases, 1):
        got = solution(words)
        if should_exist:
            assert got != "", f'case {i} ({name}): expected a valid ordering, got empty string'
            assert _is_valid_order(words, got), f'case {i} ({name}): invalid ordering {got!r}'
        else:
            assert got == "", f'case {i} ({name}): expected empty string, got {got!r}'



In [48]:
def current_solution(words):
    return Solution().alienOrder(words)


# When Solution().alienOrder is runnable, replace the two lines above with:
test(current_solution)
print("PASS")



i: 1, words[i-1]: wrt, words[i]: wrf, prev: t, curr: f
i: 2, words[i-1]: wrf, words[i]: er, prev: w, curr: e
i: 3, words[i-1]: er, words[i]: ett, prev: r, curr: t
i: 4, words[i-1]: ett, words[i]: rftt, prev: e, curr: r
graph: {'t': {'f'}, 'w': {'e'}, 'r': {'t'}, 'e': {'r'}}
indegrees: {'w': 0, 'r': 1, 't': 1, 'f': 1, 'e': 1}
Initial queue: deque(['w'])
queue: deque(['w'])
queue: deque(['e'])
queue: deque(['r'])
queue: deque(['t'])
queue: deque(['f'])
final tentative: wertf, graph: {}
i: 1, words[i-1]: z, words[i]: x, prev: z, curr: x
graph: {'z': {'x'}}
indegrees: {'z': 0, 'x': 1}
Initial queue: deque(['z'])
queue: deque(['z'])
queue: deque(['x'])
final tentative: zx, graph: {}
i: 1, words[i-1]: z, words[i]: x, prev: z, curr: x
i: 2, words[i-1]: x, words[i]: z, prev: x, curr: z
graph: {'z': {'x'}, 'x': {'z'}}
indegrees: {'z': 1, 'x': 1}
Initial queue: deque([])
final tentative: , graph: {'z': {'x'}, 'x': {'z'}}
graph: {}
indegrees: {'a': 0, 'b': 0, 'c': 0}
Initial queue: deque(['a', 'b

AssertionError: case 4 (invalid prefix): expected empty string, got 'abc'